In [ ]:
"""
MASTER EQUATION ENGINE — Modular JAX Implementation
=====================================================
χ = ∭(G · M · E · S · T · K · R · Q · F · C) dx dy dt

This is the core engine. Components are pluggable.
Disable any component → it becomes 1.0 (neutral for multiplication).
Swap any component → replace its evaluate() function.

David Lowe (POF 2828) | March 2026
"""

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit, jacfwd
import numpy as np

In [ ]:
from config import (
    N_VARS, VAR_SHORT, SYMMETRY_PAIRS, PAIR_LABELS,
    PAIR_STRENGTH, SEED_PRIMARY, SEED_SECONDARY, EPSILON_UV,
)

In [ ]:
class MasterEquation:
    """
    The full Master Equation: χ = ∭(G·M·E·S·T·K·R·Q·F·C) dx dy dt

    All 10 components are modular. Disable any subset. Swap implementations.
    Everything is JAX-differentiable.
    """

    def __init__(self, components):
        """
        Args:
            components: dict of {short_key: ComponentBase instance}
                        e.g., {"G": GraceComponent(), "M": MatterComponent(), ...}
        """
        self.components = components
        self.var_order = list(VAR_SHORT)  # fixed ordering

        # Build kinetic matrix for Lagrangian
        weights = jnp.array([
            components[k].weight if k in components else 1.0
            for k in self.var_order
        ])
        K = jnp.diag(weights)
        for (a, b) in SYMMETRY_PAIRS:
            K = K.at[a, b].set(PAIR_STRENGTH)
            K = K.at[b, a].set(PAIR_STRENGTH)
        self.kinetic_matrix = K

    def chi(self, q, t=0.0):
        """
        Compute χ from the 10 variables using modular components.

        Args:
            q: jnp array of shape (10,) — the 10 variable values
            t: time parameter

        Returns:
            scalar χ value
        """
        product = 1.0
        for i, key in enumerate(self.var_order):
            comp = self.components.get(key)
            if comp is not None and comp.enabled:
                product = product * comp.evaluate(q[i], t)
            # disabled component contributes 1.0 (neutral)
        return product

    def chi_with_breakdown(self, q, t=0.0):
        """Return chi and each component's contribution."""
        contributions = {}
        product = 1.0
        for i, key in enumerate(self.var_order):
            comp = self.components.get(key)
            if comp is not None and comp.enabled:
                val = comp.evaluate(q[i], t)
                contributions[key] = float(val)
                product = product * val
            else:
                contributions[key] = 1.0
        contributions["chi_total"] = float(product)
        return product, contributions

    def chi_4term(self, q, t=0.0):
        """
        Alternative 4-term expanded form (from existing verify_master_equation.py):
          Term 1: Grace/Entropy-Sin ratio
          Term 2: Quantum-Coherence exponential coupling
          Term 3: Faith network (observation strength)
          Term 4: Trinitarian entanglement (T-K-C triangle)
        """
        G, M, E, S, T, K, R, Q, F, C = q
        eps = EPSILON_UV

        term1 = (G * M * E) / (S + eps)
        term2 = jnp.exp(-Q * C / 10.0)
        term3 = F * (1.0 + jnp.tanh(R - 0.5))
        term4 = jnp.sqrt(jnp.abs(T * K * C) + eps)
        time_mod = 1.0 + 0.1 * jnp.sin(t) * S

        return term1 * term2 * term3 * term4 * time_mod

    def lagrangian(self, q, q_dot, t=0.0):
        """
        Lowe Coherence Lagrangian: LLC = χ(t)(dΣ/dt)² − S·χ(t)

        Args:
            q: jnp array (10,) — variable values
            q_dot: jnp array (10,) — time derivatives
            t: time
        """
        chi_val = self.chi(q, t)
        kinetic = 0.5 * q_dot @ self.kinetic_matrix @ q_dot
        potential = q[3] * chi_val  # S * chi
        return chi_val * kinetic - potential

    def equations_of_motion(self, q, q_dot, t=0.0):
        """Euler-Lagrange equations via autodiff."""
        def L_of_qdot(qd):
            return self.lagrangian(q, qd, t)
        def L_of_q(qq):
            return self.lagrangian(qq, q_dot, t)

        dL_dq = grad(L_of_q)(q)
        mass_matrix = jacfwd(grad(L_of_qdot))(q_dot)
        cond = float(jnp.linalg.cond(mass_matrix))
        q_ddot = jnp.linalg.solve(mass_matrix, dL_dq)
        return q_ddot, cond

    def energy(self, q, q_dot, t=0.0):
        """Hamiltonian H = p·q_dot - L"""
        L = self.lagrangian(q, q_dot, t)
        p = grad(lambda qd: self.lagrangian(q, qd, t))(q_dot)
        return float(jnp.dot(p, q_dot) - L)

    def mass_matrix(self, q, q_dot, t=0.0):
        """Compute and analyze the mass matrix M_ij = d^2L/dq_dot_i dq_dot_j."""
        def L_of_qdot(qd):
            return self.lagrangian(q, qd, t)
        M = jacfwd(grad(L_of_qdot))(q_dot)
        eigvals = jnp.linalg.eigvalsh(M)
        rank = int(jnp.sum(jnp.abs(eigvals) > 1e-10))
        return M, eigvals, rank

    def integrate_rk4(self, q0, q_dot0, t_span, dt=0.005):
        """4th-order Runge-Kutta integrator for the full state."""
        t0, t1 = t_span
        n_steps = int((t1 - t0) / dt)
        state = jnp.concatenate([q0, q_dot0])
        times = []
        states = []

        t = t0
        for _ in range(n_steps):
            times.append(t)
            states.append(np.array(state))

            def deriv(s, t_val):
                qq, qqd = s[:N_VARS], s[N_VARS:]
                try:
                    qacc, _ = self.equations_of_motion(qq, qqd, t_val)
                    return jnp.concatenate([qqd, qacc])
                except Exception:
                    return jnp.zeros(2 * N_VARS)

            k1 = deriv(state, t)
            k2 = deriv(state + 0.5*dt*k1, t + 0.5*dt)
            k3 = deriv(state + 0.5*dt*k2, t + 0.5*dt)
            k4 = deriv(state + dt*k3, t + dt)

            state = state + (dt/6.0) * (k1 + 2*k2 + 2*k3 + k4)
            q_new = jnp.clip(state[:N_VARS], 0.01, None)
            state = jnp.concatenate([q_new, state[N_VARS:]])
            t += dt

        return np.array(times), np.array(states)

    def gradient_chi(self, q, t=0.0):
        """Gradient of chi w.r.t. all 10 variables."""
        return grad(lambda qq: self.chi(qq, t))(q)

    def hessian_chi(self, q, t=0.0):
        """Full 10x10 Hessian of chi."""
        return jacfwd(grad(lambda qq: self.chi(qq, t)))(q)

    def sensitivity_sweep(self, q0, var_index, sweep_values):
        """
        Sweep one variable through a range, holding others fixed.
        Returns chi values and gradients at each point.
        """
        results = []
        for val in sweep_values:
            q = q0.at[var_index].set(val)
            chi_val = float(self.chi(q))
            grad_val = float(self.gradient_chi(q)[var_index])
            results.append({
                "value": float(val),
                "chi": chi_val,
                "gradient": grad_val,
            })
        return results

    def disable_component(self, key):
        """Disable a component (makes it contribute 1.0)."""
        if key in self.components:
            self.components[key].enabled = False

    def enable_component(self, key):
        """Re-enable a component."""
        if key in self.components:
            self.components[key].enabled = True

    def component_report(self):
        """Print status of all components."""
        lines = []
        for key in self.var_order:
            comp = self.components.get(key)
            if comp:
                status = "ON" if comp.enabled else "OFF"
                lines.append(f"  [{status}] {key}: {comp.name} (w={comp.weight}) <-> {comp.symmetry_partner}")
            else:
                lines.append(f"  [MISSING] {key}")
        return "\n".join(lines)